# Paper → Quiz — a typed two-stage pipeline with context primitives

Turn a research paper into a validated, concept-tagged multiple-choice quiz.
Two agents run in sequence, and every model output is **schema-validated**
before the next stage consumes it — no JSON-parsing string surgery anywhere.

```
paper.md ──▶ Stage 1: digest agent ──▶ PaperDigest ──▶ Stage 2: quiz agent ──▶ Quiz ──▶ quiz.json
```

Before running: `cp .env.example .env` and add your provider key.

In [ ]:
import json
import os
from pathlib import Path

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from vidbyte import BaseAgent, ContextManager
from vidbyte.context.primitives import FileContextItem, TaskContextItem

load_dotenv()

PROVIDER = os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai")
MODEL = os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1")
PAPER_PATH = Path("sample_paper.md")  # swap in your own markdown / plain-text paper

## Define the schemas

Both stages pass a Pydantic model as `output_schema`; the validated result
arrives on `reply.metadata["structured"]`. Stage 2 consumes Stage 1's typed
digest, so the pipeline is composable.

In [ ]:
class Claim(BaseModel):
    statement: str = Field(description="One central claim the paper makes, in plain language.")
    evidence: str = Field(description="The evidence or experiment supporting this claim.")
    caveat: str = Field(default="", description="Scope limits or conditions under which the claim holds.")


class PaperDigest(BaseModel):
    title: str = Field(description="The paper's title.")
    claims: list[Claim] = Field(description="The 3-6 most important claims.")
    key_terms: dict[str, str] = Field(description="Technical terms mapped to one-sentence definitions.")
    main_finding: str = Field(description="The single most important takeaway, in two sentences or less.")


class QuizItem(BaseModel):
    question: str = Field(description="A question testing understanding, not surface recall of phrasing.")
    options: list[str] = Field(description="Exactly four answer options.")
    answer_index: int = Field(description="0-based index of the correct option.")
    explanation: str = Field(description="Why the correct answer is right and the strongest distractor is wrong.")
    concept: str = Field(description="The concept or claim from the digest this item tests.")


class Quiz(BaseModel):
    title: str
    items: list[QuizItem] = Field(description="5-8 quiz items covering the digest's claims and terms.")


def structured(reply, schema: type[BaseModel]):
    """Pull the schema-validated payload off the reply, revalidating if the
    runtime handed back a plain dict."""
    payload = reply.metadata.get("structured")
    if payload is None:
        raise RuntimeError(f"Model output failed {schema.__name__} validation: {reply.content[:200]}")
    return payload if isinstance(payload, schema) else schema.model_validate(payload)

## Stage 1 — digest the paper

The paper enters the agent's context as a structured `FileContextItem`, and
the job description as a `TaskContextItem`, instead of being pasted into a
prompt string. Digesting first yields better quizzes than one-shot
generation: questions end up targeting the paper's actual claims and caveats
rather than surface phrasing — and the digest is a reusable artifact in its
own right.

In [ ]:
digester = BaseAgent(
    name="paper-digester",
    system_prompt=(
        "You distill research papers into faithful structured digests. "
        "Extract only what the paper actually says; record caveats honestly. "
        "Never inflate findings beyond their evidence."
    ),
    provider=PROVIDER,
    model_name=MODEL,
    context_manager=ContextManager([
        TaskContextItem(goal="Produce a faithful structured digest of the attached paper."),
        FileContextItem.from_path(str(PAPER_PATH), include_content=True),
    ]),
    output_schema=PaperDigest,
)

reply = digester.run("Digest the paper in your context into the required schema.")
digest = structured(reply, PaperDigest)

print(f"{len(digest.claims)} claims, {len(digest.key_terms)} key terms")
print("Main finding:", digest.main_finding)
for claim in digest.claims:
    print(" -", claim.statement)

## Stage 2 — write the quiz

The quiz writer never sees the raw paper — only the typed digest. Each item
carries the concept it tests, which is exactly what a spaced-repetition
system (see [`../study-agent`](../study-agent/)) needs to schedule it against
a learner's gap map.

In [ ]:
quiz_writer = BaseAgent(
    name="quiz-writer",
    system_prompt=(
        "You write retrieval-practice quizzes from paper digests. Questions must "
        "test understanding and application of claims, not memorization of wording. "
        "Distractors must be plausible misconceptions, not obvious throwaways. "
        "Tag every item with the concept it tests."
    ),
    provider=PROVIDER,
    model_name=MODEL,
    output_schema=Quiz,
)

reply = quiz_writer.run("Write a quiz from this digest:\n\n" + digest.model_dump_json(indent=2))
quiz = structured(reply, Quiz)

for i, item in enumerate(quiz.items, 1):
    print(f"{i}. [{item.concept}] {item.question}")
    for j, option in enumerate(item.options):
        marker = "✓" if j == item.answer_index else " "
        print(f"   {marker} {option}")
    print()

## Save — and optionally publish to Vidbyte

If `VIDBYTE_API_URL` / `VIDBYTE_API_KEY` are set, the quiz is POSTed to the
Vidbyte platform so it gets a hosted, shareable page and feeds the
learner-state scheduler.

In [ ]:
Path("quiz.json").write_text(quiz.model_dump_json(indent=2), encoding="utf-8")
print(f"{len(quiz.items)} items written to quiz.json")

api_url, api_key = os.getenv("VIDBYTE_API_URL", "").rstrip("/"), os.getenv("VIDBYTE_API_KEY", "")
if api_url and api_key:
    resp = requests.post(
        f"{api_url}/v1/quizzes",
        json=quiz.model_dump(),
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=30,
    )
    resp.raise_for_status()
    print("Published to Vidbyte:", resp.json().get("url", "(no url returned)"))

## Adapt it

- Swap the Pydantic schemas — cloze deletions, free-recall prompts, or exam
  blueprints are one schema change away.
- Run Stage 1 over a whole folder of PDFs (parse with `docling` or
  `markitdown` first) to build a course-sized question bank.
- Feed `PaperDigest.key_terms` into a glossary pipeline.